Published on August 05, 2026. By Prata, Marília (mpwolke)

### Don't run that 1st snippet: it will freeze your Kaggle Notebook

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

#Two lines Required to Plot Plotly
import plotly.io as pio
pio.renderers.default = 'iframe'

import plotly.graph_objs as go
import plotly.offline as py
import plotly.express as px

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## RSNA Knee Abnormality Detection

"A single knee scan can reveal a dozen different problems. In this competition, you are tasked to build machine learning models that detect a defined set of clinically important abnormalities on knee MRI examinations."

## Competition citation: 

@misc{rsna-knee-abnormality-detection,

    author = {Po-Hao “Howard” Chen and Naveen Subhas and Robyn Ball and Pieter Baeyens and Errol Colak and Ali Emami and Hillary Garner and Jacob Kazam and Hui-Ming Lin and Luciano Prevedello and Daniel Schneider and Jason Sho and Ryan Holbrook and María Cruz},
    
    title = {RSNA Knee Abnormality Detection},
    year = {2026},
    
    howpublished = {\url{https://kaggle.com/competitions/rsna-knee-abnormality-detection}},
    note = {Kaggle}
}

In [ ]:
import kagglehub

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import polars as pl
import glob
import os 
import pydicom
import matplotlib.patches as patches

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Load train and train_series

In [ ]:
tr = pd.read_csv("../input/competitions/rsna-knee-abnormality-detection/train.csv")
trs= pd.read_csv("../input/competitions/rsna-knee-abnormality-detection/train_series.csv")

## test: No more data to show (3 rows)

In [ ]:
test = pd.read_csv('../input/competitions/rsna-knee-abnormality-detection/test.csv')
test.head(4)

### sample_submission

In [ ]:
sub = pd.read_csv('../input/competitions/rsna-knee-abnormality-detection/sample_submission.csv')
sub.head(4)

In [ ]:
trs.head(2)

### Fluid and Fat issues

Fluid_Sensitive - 1 if the sequence emphasizes fluid signal (T2, PD, STIR, and similar), 0 otherwise.

Fat_Suppression - 1 if the sequence applies fat suppression, 0 otherwise.

Anatomical_Plane - imaging plane: Sagittal, Coronal, or Axial.

https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data

In [ ]:
#Chekhlov Dmitrii https://www.kaggle.com/code/zoli800/student-health-risk-balanced-accuracy-vibecode

sequence_cols = ["Fluid_Sensitive", "Fat_Suppression"]


#fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(sequence_cols):
    tab = (
        trs.groupby(col, dropna=False)['Anatomical_Plane']
        .value_counts(normalize=True)
        .rename("share")
        .reset_index()
    )
    sns.barplot(data=tab, x=col, y="share", hue='Anatomical_Plane', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Anatomical Plane")
    axes[i].tick_params(axis="x", rotation=20)
    axes[i].yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
plt.suptitle("Fat and Fluid Issues", y=1.02, fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### Anatomical_Plane - imaging plane: Sagittal, Coronal, or Axial.

In [ ]:
#By Pedro Andrade https://www.kaggle.com/code/pbizil/datahackers-managers-radiografia-dos-gestores

anat_plane_counts = trs["Anatomical_Plane"].value_counts().head(4)#Try different values of head
sns.set(style="white")
plt.figure(figsize=(6, 4))
#x=loc_counts.index, y=loc_counts.values
ax = sns.barplot(x=anat_plane_counts.index, y=anat_plane_counts.values, color=sns.color_palette("Reds", n_colors=5)[3])
plt.title("Anatomic Plane Distribution", fontsize=16)
plt.xlabel("Tags", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.xticks(rotation=45, fontsize=11)
plt.yticks(fontsize=11)
sns.despine()

#+2 is good if chart is vertical. +20 worked for horizontal
for i, v in enumerate(anat_plane_counts.values):
    ax.text(i, v + 2, str(v), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
tr.tail(3)

## info() method

In [ ]:
tr.info()

In [ ]:
tr.describe().loc[['mean','min','max']].T

### Numerical features

Numerical features to plot the histograms (heatmap not worthy)

In [ ]:
#Numerical List
#only Integer: list(df.select_dtypes(include='int64').columns)
#Below could be df.select_dtypes(include=[np.number])

list(tr.select_dtypes(include=['int64', 'float64']).columns)

In [ ]:
numerical_cols = ['ACL',
 'MCL',
 'Medial Meniscus',
 'Lateral Meniscus',
 'Medial OA',
 'Lateral OA',
 'PF OA',
 'Effusion',
 'Synovitis',
 "Baker's",
 'Contusion',
 'Fracture']

### Twelve binary labels:

ACL - anterior cruciate ligament injury (0/1).

MCL - medial collateral ligament injury (0/1).

Medial Meniscus - medial meniscus tear (0/1).

Lateral Meniscus - lateral meniscus tear (0/1).

Medial OA - osteoarthritis of the medial tibiofemoral compartment (0/1).

Lateral OA - osteoarthritis of the lateral tibiofemoral compartment (0/1).

PF OA - patellofemoral osteoarthritis (0/1).

Effusion - joint effusion / excess fluid (0/1).

Synovitis - inflammation of the joint lining (0/1).

Baker's - Baker's cyst (0/1).

Contusion - bone contusion / bone bruise (0/1).

Fracture - fracture (0/1).

### Histograms

In [ ]:
#Original figsize 15,10
tr[numerical_cols].hist(figsize=(20,15), bins=30, color='Green', edgecolor='black')
plt.suptitle("Histogram of Numeric Features")
plt.show()

### Baker's Cyst

"A Baker cyst is a fluid-filled growth behind the knee. It causes a bulge and a feeling of tightness. Also called a popliteal (pop-luh-TEE-ul) cyst, a Baker cyst can be painful. The pain can get worse with activity or when fully straightening or bending the knee."

"A Baker cyst typically is caused by an issue with the knee joint, such as arthritis or a cartilage tear. Both conditions can cause the knee to produce **too much synovial fluid**. This fluid lubricates joints to reduce friction for pain-free motion. But the fluid can build up and move into a bursa, which is a small, fluid-filled sac that cushions the joint and reduces friction. In the case of a Baker cyst, the fluid collects behind the knee, causing it to swell and form a cyst."

"While a Baker cyst may result in swelling and discomfort, treating the root cause — such as managing arthritis or repairing cartilage damage — typically leads to improvement."

https://www.mayoclinic.org/diseases-conditions/bakers-cyst/symptoms-causes/syc-20369950

In [ ]:
#By Shivam811 https://www.kaggle.com/code/shivams811/sms-spam-detection-97-67-acc-1-0-ps/notebook

sns.histplot(tr[tr["Baker's"] == 0])
sns.histplot(tr[tr["Baker's"] == 1], color='red')
#plt.legend(loc=(0, 0.7))
plt.title("Presence of Baker Cyst");

### Baker's Cyst Presence

In [ ]:
baker = tr[(tr["Baker's"]==1)].reset_index(drop=True)
baker.tail(20)

## len dcm images

In [ ]:
#Code by Abdul Basit https://www.kaggle.com/code/abdulbasitniazi/enetb7-explained-98-fine-tuning-eda

train_series = glob.glob('../input/competitions/rsna-knee-abnormality-detection/train_series/**/**/*.dcm')
print("Total number of images: ", len(train_series))

train_series = pd.Series(train_series)

In [ ]:
#By Marco Vasquez E https://www.kaggle.com/code/marcovasquez/basic-eda-data-visualization/notebook

fig=plt.figure(figsize=(15, 10))
columns = 5; rows = 4
for i in range(1, columns*rows +1):
    ds = pydicom.dcmread(train_series[i])#Original was dcmread(train_images_dir + train_images[i])
    fig.add_subplot(rows, columns, i)
    plt.imshow(ds.pixel_array, cmap=plt.cm.bone)
    fig.add_subplot

### Image type file

In [ ]:
print(ds) # this is file type of image

In [ ]:
im = ds.pixel_array
print(type(im))
print(im.dtype)
print(im.shape)

In [ ]:
#Marco Vasquez E https://www.kaggle.com/code/marcovasquez/basic-eda-data-visualization/notebook

import pylab

pylab.imshow(im, cmap=pylab.cm.gist_gray)
pylab.axis('on');

## test images

In [ ]:
test_path = '../input/competitions/rsna-knee-abnormality-detection/test_series/'

In [ ]:
test_dirs = os.listdir(test_path)

### Print the 5 first Dcm filenames

In [ ]:
# Print out the first 5 file names to verify we're in the right folder.
print (f'Total of {len(test_dirs)} DICOM images.\nFirst 5 filenames:' ) #We have only 3
test_dirs[:5]

In [ ]:
#Code by Michael Beregov https://www.kaggle.com/code/boojum/connecting-voxel-spaces

plt.figure(figsize=(14,7))
plt.subplot(121)
plt.imshow(pydicom.dcmread(f'{test_path + test_dirs[0]}/1.2.826.0.1.3680043.8.498.10543708582570884867685900269888836671/1.2.826.0.1.3680043.8.498.10044643533610889877172166379941930726.dcm').pixel_array)

plt.subplot(122)
plt.imshow(pydicom.dcmread(f'{test_path + test_dirs[0]}/1.2.826.0.1.3680043.8.498.10543708582570884867685900269888836671/1.2.826.0.1.3680043.8.498.10111435221550102244038429918428703761.dcm').pixel_array);

## Draft Session: Unfortunately, after 2h:57m I couldn't display any Baker's Cyst dcm

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRmlGgVKU8JYr-Tvmz0fQOpJtIOVMiO8FfXgRVwnQ11vTJLSsu3VUVFnR3u&s=10)

#Acknowledgements:

Michael Beregov https://www.kaggle.com/code/boojum/connecting-voxel-spaces

Chekhlov Dmitrii https://www.kaggle.com/code/zoli800/student-health-risk-balanced-accuracy-vibecode

Marco Vasquez E https://www.kaggle.com/code/marcovasquez/basic-eda-data-visualization/notebook

Abdul Basit https://www.kaggle.com/code/abdulbasitniazi/enetb7-explained-98-fine-tuning-eda

Pedro Andrade https://www.kaggle.com/code/pbizil/datahackers-managers-radiografia-dos-gestores

mpwolke https://www.kaggle.com/code/mpwolke/ia-rsna-2025-dcm-nii